# MeSH search

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

root_dir = Path.home() / 'scratch/projects/befunddolmetscher'

In [ ]:
import sys

sys.path.append((root_dir / 'search').as_posix())

Download MeSH data and unpack them:

- `en`: https://www.nlm.nih.gov/databases/download/mesh.html
  - desc2023.xml
  - supp2023.xml    
- `de`: https://repository.publisso.de/resource/frl:6473340
  - MeSH_Ger_2023_from_2023.03.21.xml

In [ ]:
import re
import unicodedata
import pandas as pd
from tqdm.auto import tqdm
from bs4 import BeautifulSoup

def normalize(s):
    return unicodedata.normalize('NFKD', s).strip()

for lang, filename in [('en', 'desc2023.xml'), ('de', 'MeSH_Ger_2023_from_2023.03.21.xml')]:

    xml = (root_dir / 'data' / filename).read_text()
    soup = BeautifulSoup(xml, features="xml")
    
    records = []
    for desc in tqdm(soup.find_all('DescriptorRecord'), desc=f"parse {lang} MeSH xml"):
        desc_ui = desc.find('DescriptorUI').text.strip()
        desc_name = desc.find('DescriptorName').text.strip()
        for concept in desc.find_all('Concept'):
            concept_ui = concept.find('ConceptUI').text.strip()
            concept_name = concept.find('ConceptName').text.strip()
            scope_note = desc.find('ScopeNote')
            preferred_concept = concept['PreferredConceptYN']
            for term in concept.find_all('Term'):
                term_name = re.sub(r'\[[^\]]+\]$', '', normalize(term.find('String').text))
                term_ui = term.find('TermUI').text.strip()
                if lang == 'en' or (lang == 'de' and term_ui.startswith('ger')):
                    records.append({
                        'descriptor_ui': desc_ui,
                        f'descriptor_{lang}': re.sub(r'\[[^\]]+\]$', '', normalize(desc_name)),
                        'scope_note': normalize(scope_note.text) if scope_note else None,
                        'concept_ui': concept_ui,
                        'preferred_concept_yn': preferred_concept,
                        f'concept_{lang}': re.sub(r'\[[^\]]+\]$', '', normalize(concept_name)),
                        'term_ui': term_ui,
                        'term': term_name,
                        'concept_preferred_term_yn': term['ConceptPreferredTermYN'],
                        'is_permuted_term_yn': term['IsPermutedTermYN'],
                        'lexical_tag': term['LexicalTag'],
                        'record_preferred_term_yn': term['RecordPreferredTermYN'],
                    })
    df = pd.DataFrame.from_records(records)
    df.to_csv(root_dir / f'data/MeSH_2023_{lang}.tsv', sep="\t", index=False)

In [ ]:
import pandas as pd

dfs = {
    'en': pd.read_csv(root_dir / f'data/MeSH_2023_en.tsv', sep="\t", index_col=False).fillna(''),
    'de': pd.read_csv(root_dir / f'data/MeSH_2023_de.tsv', sep="\t", index_col=False).fillna(''),
}
len(dfs['en']), len(dfs['de'])

In [ ]:
for lang, df in dfs.items():
    other_lang = 'de' if lang == "en" else 'en'
    if f'descriptor_{other_lang}' not in df.columns:
        descriptor = dfs[other_lang][['descriptor_ui', f'descriptor_{other_lang}']].set_index('descriptor_ui').to_dict()[f'descriptor_{other_lang}']
        concept = dfs[other_lang][['concept_ui', f'concept_{other_lang}']].set_index('concept_ui').to_dict()[f'concept_{other_lang}']
        df[f'descriptor_{other_lang}'] = df['descriptor_ui'].apply(lambda x: descriptor.get(x))
        df[f'concept_{other_lang}'] = df['concept_ui'].apply(lambda x: concept.get(x))
        header = [
            'term_ui', 'term', 'descriptor_ui', f'descriptor_{lang}', f'descriptor_{other_lang}',
            'scope_note', 'concept_ui', 'preferred_concept_yn', f'concept_{lang}', f'concept_{other_lang}',
            'concept_preferred_term_yn', 'is_permuted_term_yn', 'lexical_tag', 'record_preferred_term_yn'
        ]
        df[header].to_csv(root_dir / f'data/MeSH_2023_{lang}.tsv', sep="\t", index=False)

parse supplemental material

Note: Desease Onthology partly maps MeSH Supplemental Record UI <-> ICD 10 CM

In [ ]:
import json
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

xml = (root_dir / 'data/supp2023.xml').read_text()
soup = BeautifulSoup(xml, features="xml")
supp_uis = {}
for supp in tqdm(soup.find_all('SupplementalRecord')):
    supp_ui = supp.find('SupplementalRecordUI').text
    desc = [d.find('DescriptorUI').text.lstrip('*').strip() for d in supp.find_all('DescriptorReferredTo')]
    concept = [c.find('ConceptUI').text.strip() for c in supp.find_all('ConceptList')]
    term = [t.find('TermUI').text.strip() for t in supp.find_all('TermList')]
    note = supp.find('Note')
    supp_uis[supp_ui] = {
        'descriptor_ui': desc,
        'concept_ui': concept,
        'term_ui': term,
        'note': normalize(note.text) if note is not None else None,
    }
with (root_dir / 'data/MeSH_supp2023.json').open('w') as f:
    json.dump(supp_uis, f, indent=4)

In [ ]:
from tqdm.auto import tqdm

def aggregate(df, lang):
    df = df[(df.is_permuted_term_yn == 'N') | (~df.term.str.contains(', '))]
    records = []
    for descriptor_ui, group in tqdm(df.groupby('descriptor_ui')):
        descriptor_en = '; '.join(set(group['descriptor_en'].tolist()))
        descriptor_de = '; '.join(set(group['descriptor_de'].tolist()))
        concept_names = '; '.join(set(group[f'concept_{lang}'].tolist()))
        terms = '; '.join(set(group.term.tolist()))
        scope_notes = '; '.join(set(group.scope_note.tolist()))
        descriptor_names = descriptor_en if lang == 'en' else descriptor_de
        text = '\n\n'.join([descriptor_names, concept_names, terms, scope_notes])
        records.append({
            'descriptor_ui': descriptor_ui,
            'descriptor_en': descriptor_en,
            'descriptor_de': descriptor_de,
            'text': text,
            'synonyms': terms,
        })
    return pd.DataFrame.from_records(records)

In [ ]:
descriptors_df = {lang: aggregate(df, lang) for lang, df in dfs.items()}

In [ ]:
import requests
def embed(contents: list, host: str="localhost", port: str="8080"):
    """embed texts"""
    response = requests.post(
        f"http://{host}:{port}/embedding",
        headers={"Content-Type": "application/json"},
        json={"content": contents},
    )
    return response.json()

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import json
output_path = (root_dir / 'results/mesh_embed.jsonl')

for lang, df in descriptors_df.items():
    df = df.fillna('')
    batch_size = 64
    for key, group in tqdm(df.groupby(np.arange(len(df))//batch_size), total=(len(df)//batch_size +1)):
        desc_ui = group.descriptor_ui.to_list()
        texts = group.text.to_list()
        assert len(texts) == len(desc_ui)
    
        vectors = embed(contents=texts, host='localhost', port='8181')
        assert len(vectors) == len(texts)
    
        # save in json file
        for v, ui in zip(vectors, desc_ui):
            with output_path.open('a') as f:
                v['descriptor_ui'] = ui
                v['language'] = lang
                f.write(json.dumps(v) + '\n')

In [ ]:
import json

output_path = (root_dir / 'results/mesh_embed.jsonl')

vectors = [json.loads(l) for l in output_path.read_text().splitlines()]
vectors = {(v['descriptor_ui'], v['language']): v['embedding'][0] for v in vectors if len(v['embedding'][0]) == 4096}
len(vectors)

In [ ]:
from tqdm.auto import tqdm

metadatas = {'en': [], 'de': []}
embeds = {'en': [], 'de': []}
for lang, df in descriptors_df.items():
    for name, row in tqdm(df.iterrows()):
        metadatas[lang].append({
            'descriptor_ui': row['descriptor_ui'],
            'german' if lang == 'en' else 'english': row['descriptor_de'] if lang == 'en' else row['descriptor_en'],
        })
        embeds[lang].append(vectors[(row['descriptor_ui'], lang)])

In [ ]:
from sqlitevec import SQLiteVec

mesh_table = {}
for lang, df in descriptors_df.items():
    mesh_table[lang] = SQLiteVec.from_texts(
        titles=df[f'descriptor_{lang}'].to_list(),
        texts=df.text.to_list(),
        embeds=embeds[lang],
        metadatas=metadatas[lang],
        table=f"mesh_{lang}",
        db_file="mesh.db",
        lang=lang,
    )

In [ ]:
found = mesh_table['en'].metadata_search(
    descriptor_ui=['D001064'] #exact match
)
for f in found:
    print(f['descriptor_ui'], f['title'], f['german'])

In [ ]:
found = mesh_table['de'].metadata_search(
    descriptor_ui=['D001064'] #exact match
)
for f in found:
    print(f['descriptor_ui'], f['title'], f['english'])

In [ ]:
from IPython.display import display, HTML

found = mesh_table['en'].fulltext_search_with_highlight(
    query='(title: Appendicitis) OR (text: Appendicitis)' #lemma match
)
for f in found: # relevance rank from BM25
    print(f['descriptor_ui'], f['german'])
    display(HTML(f['title']))
    display(HTML(f['text'])) # synonyms
    print('---')

In [ ]:
found = mesh_table['de'].fulltext_search_with_highlight(
    query='(title: Blinddarmentzündung) OR (text: Blinddarmentzündung)' #lemma match
)
for f in found: # relevance rank from BM25
    print(f['descriptor_ui'], f['english'])
    display(HTML(f['title']))
    display(HTML(f['text'])) # synonyms
    print('---')